In [ ]:
# import mne
import os
from pathlib import Path
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


In [5]:


# 1. Setup Parameters
fs = 160          # Sampling rate
window_sec = 1    # 1 second windows
window_size = fs * window_sec 

def create_time_decoding_dataset(data):
    """
    Input: data of shape (samples, channels) e.g., (9600, 64)
    Returns: X (windows, features), y (labels)
    """
    n_samples, n_channels = data.shape
    
    # Let's define "Early" as first 25 seconds, "Late" as last 25 seconds
    early_cutoff = 25 * fs
    late_start = n_samples - (25 * fs)
    
    X, y = [], []
    
    # Extract Early Windows (Class 0)
    for i in range(0, early_cutoff - window_size, window_size):
        window = data[i : i + window_size]
        # Feature extraction: use raw flattened data or Power (Mean Square)
        # Flattening 1s of 64 channels gives 10,240 features. 
        X.append(window.flatten()) 
        y.append(0)
        
    # Extract Late Windows (Class 1)
    for i in range(late_start, n_samples - window_size, window_size):
        window = data[i : i + window_size]
        X.append(window.flatten())
        y.append(1)
        
    return np.array(X), np.array(y)


# 2. Load and Process Real Data
# Ensure your data is (Time, Channels)
real_raw = np.load("real_data.npy") 
if real_raw.shape[0] < real_raw.shape[1]: real_raw = real_raw.T 

X_real, y_real = create_time_decoding_dataset(real_raw)



In [8]:

# 3. Train Decoder on Real Data
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.2, stratify=y_real)

# We use a Linear SVM - good for high-dimensional EEG features
clf = make_pipeline(StandardScaler(), SVC(kernel='linear'))
clf.fit(X_train, y_train)

real_acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Decoding Accuracy (Real Data): {real_acc:.2f}")

Decoding Accuracy (Real Data): 0.30


In [6]:
25*fs

4000

In [7]:
print(y_real)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1]


In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

def create_channel_wise_dataset(data, fs=160, window_sec=1):
    """
    data: (samples, channels)
    """
    n_samples, n_channels = data.shape
    win_len = fs * window_sec
    
    # Define Early (first 25s) and Late (last 25s)
    early_data = data[:25*fs, :]
    late_data = data[-25*fs:, :]
    
    X, y = [], []
    
    # Process Early (Label 0)
    for ch in range(n_channels):
        ch_signal = early_data[:, ch]
        for i in range(0, len(ch_signal) - win_len, win_len):
            window = ch_signal[i : i + win_len]
            X.append(window) # Feature is the 1s segment of 1 channel
            y.append(0)
            
    # Process Late (Label 1)
    for ch in range(n_channels):
        ch_signal = late_data[:, ch]
        for i in range(0, len(ch_signal) - win_len, win_len):
            window = ch_signal[i : i + win_len]
            X.append(window)
            y.append(1)
            
    return np.array(X), np.array(y)

# 1. Load your data
real_raw = np.load("real_data.npy") # Ensure shape is (9600, 64)
gen_raw = np.load("rnn_generated_data.npy")

# 2. Prepare datasets
X_real, y_real = create_channel_wise_dataset(real_raw)
X_gen, y_gen = create_channel_wise_dataset(gen_raw)

# 3. Define the Decoder
# Using RBF kernel since we have more samples now and 1D features
clf = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1.0))

# 4. Train on REAL
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.2, stratify=y_real)
clf.fit(X_train, y_train)
real_acc = accuracy_score(y_test, clf.predict(X_test))

# 5. Evaluate on GENERATED (Transfer or separate train)
# Technique: Train a NEW model on Generated data to see if the signal exists there too
clf_gen = make_pipeline(StandardScaler(), SVC(kernel='linear'))
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(X_gen, y_gen, test_size=0.2, stratify=y_gen)
clf_gen.fit(X_train_g, y_train_g)
gen_acc = accuracy_score(y_test_g, clf_gen.predict(X_test_g))

print(f"Real Data Accuracy: {real_acc:.2f}")
print(f"Generated Data Accuracy: {gen_acc:.2f}")

Real Data Accuracy: 0.95
Generated Data Accuracy: 0.87


In [12]:
print(y_test)

[1 1 1 0 0 1 0 1 0 0 1 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 1 0
 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 1 0 1 0 0 1 0 0 1 0 1 0 1 0 1 0 1 0
 0 0 1 1 1 0 1 0 1 0 0 1 0 1 0 1 1 0 0 0 1 0 1 0 1 0 1 1 1 1 1 1 1 1 1 0 1
 1 1 0 1 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 0 0 1 0 1 0 0 0 1 1 0 0 0 0 0 1 1 0
 0 1 1 0 1 1 0 0 0 0 1 0 1 0 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 1 1 1 0 1 1 1
 1 0 0 1 1 1 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0
 0 0 0 1 1 1 1 0 0 0 1 1 1 1 0 0 0 0 1 0 1 0 1 1 0 1 0 1 1 0 1 0 0 1 0 0 1
 0 0 0 1 1 1 1 1 0 0 1 1 1 0 1 1 0 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 0 0 1 0 1
 0 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 1 0 0 1 1 1 0 0 0 0 1
 1 1 0 1 0 0 0 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 0 1 0 0 1 0
 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1 0 1 1 1 1 1 1 0 1 1 0 1 0 1 0 0 0 0 1 1 0 0
 0 0 0 1 0 0 0 0 0 0 1 1 1 1 1 0 1 0 0 0 0 1 1 0 0 0 1 1 1 1 0 1 0 0 1 1 0
 0 0 1 1 0 1 1 1 0 1 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 1 1 0 0 1 0 0 1 0 0
 1 0 0 1 0 1 1 0 1 1 1 0 